# Pipeline de PLN con Programacion Funcional

Pipeline completo aplicando `map`, `filter`, `reduce` y composicion de funciones.

Las funciones puras siempre devuelven el mismo resultado dado el mismo input,
lo que facilita encadenar y reusar pasos de procesamiento.

## Ejercicio 1: Limpieza con funciones puras y `map`

**Concepto:** `map(f, iterable)` aplica una funcion a cada elemento del iterable.

1. Define funciones puras para cada paso de limpieza.
2. Compara la version con `for` vs con `map`.
3. Verifica que el resultado es identico.

In [ ]:
import re
from functools import reduce

oraciones = [
    "El Procesamiento de Lenguaje Natural (PLN) es clave en la IA.",
    "Las maquinas pueden leer, interpretar y comprender texto humano.",
    "Aplicaciones: traduccion automatica, chatbots y analisis de sentimientos.",
    "En 2023, los LLMs como GPT-4 han avanzado significativamente.",
    "Tecnicas como Word2Vec y transformers representan el lenguaje en vectores.",
    "La tokenizacion divide el texto en unidades llamadas tokens.",
    "El stemming reduce palabras a su raiz; la lematizacion a su forma base.",
    "Los modelos de lenguaje aprenden patrones estadisticos del corpus.",
]

# Funciones puras: cada una hace UNA sola cosa
def a_minusculas(t):      return t.lower()
def quitar_especiales(t): return re.sub(r'[^a-z\s]', '', t)
def colapsar_espacios(t): return re.sub(r'\s+', ' ', t).strip()

def limpiar(texto):
    return colapsar_espacios(quitar_especiales(a_minusculas(texto)))

# Version imperativa (bucle for)
limpios_imp = []
for o in oraciones:
    limpios_imp.append(limpiar(o))

# Version funcional con map
limpios_func = list(map(limpiar, oraciones))

assert limpios_imp == limpios_func
print("Ambas versiones producen el mismo resultado.")
print("\nOraciones limpias:")
for o in limpios_func:
    print(f"  {o}")


## Ejercicio 2: Composicion de funciones con `reduce`

**Concepto:** Podemos componer N funciones en una sola con `reduce`:

```python
componer([f, g, h])(x)  ==  h(g(f(x)))
```

1. Define `componer` usando `reduce`.
2. Construye el pipeline como lista de pasos.
3. Prueba agregar o quitar pasos sin cambiar el resto del codigo.

In [ ]:
def componer(funciones):
    """Devuelve una funcion que aplica cada f en orden (izq a derecha)."""
    return reduce(lambda f, g: lambda x: g(f(x)), funciones)

# El pipeline es una lista declarativa de pasos
pipeline_basico   = componer([a_minusculas, quitar_especiales, colapsar_espacios])
pipeline_completo = componer([a_minusculas, quitar_especiales, colapsar_espacios])

ejemplo = "  El PLN procesa informacion con MAYUSCULAS y signos!!!  "

print(f"Original          : {ejemplo.strip()}")
print(f"Pipeline basico   : {pipeline_basico(ejemplo)}")
print(f"Pipeline completo : {pipeline_completo(ejemplo)}")

# Aplicar el pipeline a todo el corpus con map
corpus_limpio = list(map(pipeline_completo, oraciones))
print("\nCorpus limpio:")
for r in corpus_limpio:
    print(f"  {r}")


## Ejercicio 3: Tokenizacion y stopwords con `filter`

**Concepto:** `filter(predicado, iterable)` conserva solo los elementos
donde `predicado(x)` es `True`.

1. Tokeniza con `map`.
2. Define predicados como funciones nombradas y lambdas.
3. Encadena predicados con una funcion de orden superior.

In [ ]:
import nltk
from nltk.corpus import stopwords

for ruta, nombre in [('corpora/stopwords','stopwords'),
                      ('tokenizers/punkt_tab','punkt_tab')]:
    try: nltk.data.find(ruta)
    except LookupError: nltk.download(nombre, quiet=True)

STOPWORDS = set(stopwords.words('spanish'))

# map: tokenizar cada oracion
tokens_por_oracion = list(map(str.split, corpus_limpio))

# reduce: aplanar lista de listas
todos = reduce(lambda a, b: a + b, tokens_por_oracion, [])
print(f"Total tokens: {len(todos)}")

# Predicados (funciones que devuelven bool)
no_es_stopword  = lambda t: t not in STOPWORDS
longitud_minima = lambda t: len(t) > 2
no_es_numero    = lambda t: not t.isdigit()

# Funcion de orden superior: compone predicados con AND logico
def todos_cumplen(predicados):
    return lambda x: all(p(x) for p in predicados)

predicado = todos_cumplen([no_es_stopword, longitud_minima, no_es_numero])
tokens_validos = list(filter(predicado, todos))

eliminados = len(todos) - len(tokens_validos)
print(f"Tokens eliminados: {eliminados} ({eliminados/len(todos)*100:.1f}%)")
print(f"Tokens validos   : {len(tokens_validos)}")
print(f"\nPrimeros 15: {tokens_validos[:15]}")


## Ejercicio 4: Frecuencias con `reduce` y reporte con `map`

**Concepto:** En programacion funcional las agregaciones se expresan con `reduce`.

1. Construye el diccionario de frecuencias solo con `reduce`.
2. Formatea el reporte con `map`.
3. Encuentra el token mas frecuente con `reduce`.

In [ ]:
# Frecuencias con reduce (sin Counter ni for)
def agregar(acum, token):
    acum[token] = acum.get(token, 0) + 1
    return acum

frecuencias = reduce(agregar, tokens_validos, {})
top10 = sorted(frecuencias.items(), key=lambda par: par[1], reverse=True)[:10]

# Formatear filas del reporte con map
def formatear(par):
    p, n = par
    return f"  {p:<22} {n:>4}  {'#' * n}"

lineas = list(map(formatear, top10))

print("Top 10 palabras mas frecuentes:")
for linea in lineas:
    print(linea)

# Token mas frecuente con reduce
mas_frecuente = reduce(lambda a, b: a if a[1] >= b[1] else b, frecuencias.items())
print(f"\nPalabra mas frecuente: '{mas_frecuente[0]}' ({mas_frecuente[1]} veces)")

# Longitud promedio de tokens con map y reduce
longitudes = list(map(len, tokens_validos))
promedio   = reduce(lambda a, b: a + b, longitudes) / len(longitudes)
print(f"Longitud promedio de token: {promedio:.2f} letras")


## Ejercicio 5: Generacion de texto con funcion de orden superior

**Concepto:** Una **funcion de orden superior** recibe o devuelve otras funciones.

1. `crear_generador` devuelve una funcion especializada segun el orden del modelo.
2. Usa `map` para generar multiples textos de una vez.

In [ ]:
import random
from collections import defaultdict

def construir_modelo(tokens, orden=1):
    """Construye un modelo de Markov de orden N."""
    modelo = defaultdict(list)
    for i in range(len(tokens) - orden):
        clave = tuple(tokens[i:i+orden])
        modelo[clave].append(tokens[i+orden])
    return modelo

def crear_generador(modelo, orden, longitud=8):
    """Funcion de orden superior: devuelve una funcion generadora."""
    def generar(_=None):
        clave = random.choice(list(modelo.keys()))
        resultado = list(clave)
        for __ in range(longitud - orden):
            siguientes = modelo.get(tuple(resultado[-orden:]))
            if not siguientes:
                break
            resultado.append(random.choice(siguientes))
        return ' '.join(resultado)
    return generar

random.seed(42)
mk1 = construir_modelo(tokens_validos, orden=1)
mk2 = construir_modelo(tokens_validos, orden=2)

generar_1 = crear_generador(mk1, orden=1)
generar_2 = crear_generador(mk2, orden=2)

# map aplica el generador sobre un rango auxiliar
print("Orden 1 (menos coherente):")
for texto in map(generar_1, range(3)):
    print(f"  {texto}")

print("\nOrden 2 (mas coherente):")
for texto in map(generar_2, range(3)):
    print(f"  {texto}")
